## 1. Environment Setup

In [ ]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

In [ ]:
# Install Flash Attention (optional but recommended for A100/H100)
# !pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.5.4/flash_attn-2.6.3+cu124torch2.9-cp312-cp312-linux_x86_64.whl

In [ ]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")

In [ ]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
else:
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -q -e .
!pip install -q wandb tensorboard datasets>=2.14.0 accelerate
print("Dependencies installed!")

In [ ]:
# Mount Google Drive (for persistent storage) and safely link outputs/checkpoints
import os
from datetime import datetime
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT_v3"
    drive_outputs = Path(f"{DRIVE_BASE}/outputs")
    drive_checkpoints = Path(f"{DRIVE_BASE}/checkpoints")

    # Ensure Drive directories exist
    drive_outputs.mkdir(parents=True, exist_ok=True)
    drive_checkpoints.mkdir(parents=True, exist_ok=True)

    def ensure_symlink(local_path: Path, target: Path) -> None:
        """Create a symlink from local_path -> target without deleting target data."""
        if local_path.exists():
            if local_path.is_symlink():
                current_target = Path(os.readlink(local_path))
                if current_target == target:
                    print(f"✅ {local_path} already linked to Drive")
                    return
                else:
                    print(f"🔄 Updating symlink for {local_path} -> {target}")
                    local_path.unlink()
            else:
                # Preserve any existing local data by backing it up
                backup_name = f"{local_path.name}_local_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
                backup_path = local_path.parent / backup_name
                print(f"📦 Preserving existing local {local_path} at {backup_path}")
                local_path.rename(backup_path)
        # Create symlink to Drive
        local_path.symlink_to(target, target_is_directory=True)
        print(f"🔗 Linked {local_path} -> {target}")

    # Safely link outputs and checkpoints to Drive (no deletion of Drive data)
    ensure_symlink(Path("outputs"), drive_outputs)
    ensure_symlink(Path("checkpoints"), drive_checkpoints)

    print(f"Outputs and checkpoints will be saved to: {DRIVE_BASE}")
else:
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Suppress TensorFlow warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## 2. Check Data Availability

In [ ]:
# Check data and model availability
import os
import glob
from pathlib import Path

def check_v3_requirements():
    """
    Verify requirements for v3 training.

    Phase 0.5 uses HuggingFace datasets (downloaded automatically):
        - SST-2, MNLI, CoNLL-2003, SQuAD, STS-B

    Phase 1 uses local FamilyOS data.

    Model must be initialized from v2 checkpoint.
    """

    print("="*60)
    print("CHECKING V3 TRAINING REQUIREMENTS")
    print("="*60)

    # Phase 0.5 - Uses HuggingFace datasets (NO local files needed!)
    print("\n[Phase 0.5] Healing Data:")
    print("   Uses HuggingFace datasets (auto-downloaded):")
    print("   - glue/sst2 (sentiment)")
    print("   - glue/mnli (NLI)")
    print("   - conll2003 (NER)")
    print("   - squad (QA)")
    print("   - glue/stsb (similarity)")
    print("   [OK] No local data required for Phase 0.5")

    # Phase 1 & 2 - Local FamilyOS data (from YAML: familyos_unified.yaml)
    print("\n[Phase 1 & 2] FamilyOS Unified Data (8 tasks):")
    print("   Config: configs/data/multitask/familyos_unified.yaml")
    print("   Tasks: emotions, sentiment, safety, intent, ingress, ner_family, temporal, relations")

    phase1_path = "data/familyos/unified/output_healed_merged"
    phase1_ok = os.path.exists(phase1_path)

    if phase1_ok:
        # Count shards
        shards = glob.glob(os.path.join(phase1_path, "shard_*.jsonl"))
        print(f"   [OK] {phase1_path}")
        print(f"        Found {len(shards)} shard files")
    else:
        print(f"   [MISSING] {phase1_path}")
        print("        -> Run data healing scripts first!")

    # Replay data (healing data used during Phase 1 & 2)
    print("\n[Replay] Healing data for forgetting prevention:")
    replay_path = "data/healing/healing_enhanced.jsonl"
    replay_exists = os.path.exists(replay_path)
    if replay_exists:
        size_mb = os.path.getsize(replay_path) / 1e6
        print(f"   [OK] {replay_path} ({size_mb:.1f} MB)")
    else:
        print(f"   [OPTIONAL] {replay_path}")
        print("        (Will be created from HuggingFace during Phase 0.5)")

    # v2 checkpoint (required for v3 initialization)
    # IMPORTANT: Returns the MODEL FILE path, not directory!
    print("\n[Model] v2 Checkpoint (for v3 initialization):")
    v2_paths = [
        # Best checkpoint first (step 18000 > step 6000)
        "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000/model.safetensors",
        "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000/pytorch_model.bin",
        # Root directory (older checkpoint at step 6000)
        "outputs/modernbert-v2-for-v3-transfer/model.safetensors",
        "outputs/modernbert-v2-for-v3-transfer/pytorch_model.bin",
        # Checkpoints directory
        "checkpoints/modernbert-v2-for-v3-transfer/model.safetensors",
        "checkpoints/modernbert-v2-for-v3-transfer/pytorch_model.bin",
    ]

    v2_found = None
    for file_path in v2_paths:
        if os.path.exists(file_path):
            v2_found = file_path  # Return FILE path, not directory
            break

    if v2_found:
        size_mb = os.path.getsize(v2_found) / 1e6
        print(f"   [OK] v2 checkpoint: {v2_found} ({size_mb:.1f} MB)")
    else:
        print("   [MISSING] v2 checkpoint not found!")
        print("   Checked paths:")
        for p in v2_paths:
            print(f"      - {p}")

    # v3 initialized model
    print("\n[Model] v3 Initialized Model:")
    v3_paths = [
        "checkpoints/modernbert-v3-initialized",
        "outputs/v3_full/phase_0.5/best_model",
    ]

    v3_found = None
    for path in v3_paths:
        if os.path.exists(path):
            # Check for any model file
            has_model = (
                os.path.exists(os.path.join(path, "pytorch_model.bin")) or
                os.path.exists(os.path.join(path, "model.safetensors"))
            )
            if has_model:
                v3_found = path
                break

    if v3_found:
        print(f"   [OK] v3 model: {v3_found}")
    else:
        print("   [NOT INITIALIZED] v3 model needs to be created from v2")
        if v2_found:
            print("   -> Run the 'Initialize v3 from v2' cell below")
        else:
            print("   -> First need v2 checkpoint, then initialize v3")

    print("\n" + "="*60)

    return phase1_ok, v2_found, v3_found

phase1_ready, v2_checkpoint, v3_model = check_v3_requirements()

## 3. Initialize v3 Model from v2

**This is required before training!**

The v3 model (28 layers) is created by:
1. Copying v2 layers 1-22 directly
2. Cloning v2 layers 15-20 to v3 layers 23-28 (family context layers)
3. Adding hub tokens [EMO], [MEM], [REL], [TASK]

In [ ]:
%%time

import os
from pathlib import Path

# Skip if v3 already initialized
if v3_model:
    print(f"v3 model already exists at: {v3_model}")
    print("Skipping initialization.")
else:
    print("="*60)
    print("INITIALIZING V3 MODEL FROM V2")
    print("="*60)

    # v2_checkpoint is now the FILE path (model.safetensors or pytorch_model.bin)
    if v2_checkpoint:
        v2_file = v2_checkpoint
    else:
        # Fallback: find the best checkpoint file
        candidates = [
            "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000/model.safetensors",
            "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000/pytorch_model.bin",
            "outputs/modernbert-v2-for-v3-transfer/model.safetensors",
            "outputs/modernbert-v2-for-v3-transfer/pytorch_model.bin",
        ]
        v2_file = None
        for c in candidates:
            if os.path.exists(c):
                v2_file = c
                break
        if not v2_file:
            raise FileNotFoundError("No v2 checkpoint found! Check outputs/modernbert-v2-for-v3-transfer/")

    v3_output = "checkpoints/modernbert-v3-initialized"

    print(f"v2 checkpoint file: {v2_file}")
    print(f"v3 output dir:      {v3_output}")

    # Run initialization script - pass the FILE path, not directory
    cmd = f"python -u scripts/v3_scripts/initialize_v3_from_v2.py "
    cmd += f"--v2-checkpoint {v2_file} "
    cmd += f"--output-dir {v3_output} "
    cmd += "--verify"

    print(f"\nCommand: {cmd}\n")
    !{cmd}

    # Verify initialization
    if os.path.exists(f"{v3_output}/pytorch_model.bin") or os.path.exists(f"{v3_output}/model.safetensors"):
        for ext in ["pytorch_model.bin", "model.safetensors"]:
            model_file = f"{v3_output}/{ext}"
            if os.path.exists(model_file):
                size_mb = os.path.getsize(model_file) / 1e6
                print("\n" + "="*60)
                print(f"V3 MODEL INITIALIZED SUCCESSFULLY!")
                print(f"   Path: {v3_output}")
                print(f"   File: {ext}")
                print(f"   Size: {size_mb:.1f} MB")
                print("="*60)
                break
        v3_model = v3_output  # Update for next cells
    else:
        print("\n" + "="*60)
        print("V3 INITIALIZATION FAILED!")
        print("Check logs above for errors.")
        print("="*60)

## 4. Training Configuration

In [ ]:
# Verify model chaining
print("="*60)
print("MODEL CHAINING VERIFICATION")
print("="*60)
print("""
v2 Checkpoint                    v3 Initialized              Phase 0.5
outputs/modernbert-v2-     -->   checkpoints/modernbert-  -->  outputs/v3_full/
  for-v3-transfer/                 v3-initialized/              phase_0.5/best_model/
                                                                      |
                                                                      v
Phase 2                     <--  Phase 1.5 (eval)  <--        Phase 1
outputs/v3_full/                 outputs/v3_full/             outputs/v3_full/
  phase_2/best_model/              phase_1.5/                   phase_1/best_model/
                                   (results.json)                     |
                                                                      |
                                   Phase 2 uses Phase 1 model --------+
                                   (1.5 is evaluation only)
""")
print("="*60)

In [ ]:
# Training Configuration
import os

# Output directory
OUTPUT_DIR = "outputs/v3_full"

# Debug mode: Set to True for quick testing (5 steps per phase)
DEBUG_RUN = False

# Which phases to run
RUN_PHASE_0_5 = True   # Enhanced Healing (uses HuggingFace datasets)
RUN_PHASE_1 = True     # Multi-Task FamilyOS (uses local data)
RUN_PHASE_1_5 = True   # Forgetting Evaluation
RUN_PHASE_2 = True     # Fine-Tuning

# W&B logging (disable for debug runs)
USE_WANDB = not DEBUG_RUN

# Model path (set automatically from initialization or override here)
MODEL_PATH = v3_model if v3_model else "checkpoints/modernbert-v3-initialized"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model path:       {MODEL_PATH}")
print(f"Debug mode:       {DEBUG_RUN}")
print(f"W&B logging:      {USE_WANDB}")
print(f"\nPhases to run:")
print(f"   Phase 0.5 (Healing):     {RUN_PHASE_0_5} - uses HuggingFace datasets")
print(f"   Phase 1 (Multi-Task):    {RUN_PHASE_1} - uses local FamilyOS data")
print(f"   Phase 1.5 (Forgetting):  {RUN_PHASE_1_5}")
print(f"   Phase 2 (Fine-Tuning):   {RUN_PHASE_2}")
print("="*60)

In [ ]:
# Pull latest code before training
!git pull origin main

---
## 5. Phase 0.5: Enhanced Healing

**Purpose:** Heal the cloned layers (L23-28) and establish smooth activation flow.

**Data:** HuggingFace datasets (downloaded automatically):
- SST-2: Sentiment classification
- MNLI: Natural Language Inference  
- CoNLL-2003: Named Entity Recognition
- SQuAD: Question Answering
- STS-B: Semantic Textual Similarity

**Training:** 2,500 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_0_5:
    print("="*60)
    print("PHASE 0.5: Enhanced Healing")
    print("="*60)
    print("Data: HuggingFace datasets (SST-2, MNLI, CoNLL, SQuAD, STS-B)")
    print(f"Model: {MODEL_PATH}")

    # Build command
    cmd = f"python -u scripts/v3_scripts/train_v3_phase0_5.py "
    cmd += f"--config configs/training/multitask/stage_v3_phase0_5_enhanced.yaml "
    cmd += f"--output-dir {OUTPUT_DIR}/phase_0.5 "
    cmd += f"--model-path {MODEL_PATH} "

    if DEBUG_RUN:
        cmd += "--max-steps 5 --debug "

    if not USE_WANDB:
        cmd += "--no-wandb "
    else:
        cmd += "--wandb-run-name v3_phase_0.5 "

    print(f"\nCommand: {cmd}\n")
    !{cmd}

    # Check success - Phase 0.5 saves to 'best/' not 'best_model/'
    phase_0_5_ok = (
        os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best/pytorch_model.bin") or
        os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best_model/pytorch_model.bin") or
        os.path.exists(f"{OUTPUT_DIR}/phase_0.5/final_model/pytorch_model.bin")
    )

    if phase_0_5_ok:
        print("\n" + "="*60)
        print("PHASE 0.5 COMPLETED SUCCESSFULLY!")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("PHASE 0.5 FAILED! Check logs above.")
        print("="*60)
else:
    print("Phase 0.5 skipped (RUN_PHASE_0_5 = False)")
    phase_0_5_ok = (
        os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best") or
        os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best_model")
    )

In [ ]:
# Verify Phase 0.5 output
import os
import json

phase_0_5_output = f"{OUTPUT_DIR}/phase_0.5"

if os.path.exists(phase_0_5_output):
    print(f"Phase 0.5 output: {phase_0_5_output}")
    print()

    # Check for model - Phase 0.5 saves to 'best/', not 'best_model/'
    for model_dir in ["best", "best_model", "final_model"]:
        model_path = os.path.join(phase_0_5_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model directory: {model_dir}/")
            for f in os.listdir(model_path):
                fpath = os.path.join(model_path, f)
                if os.path.isfile(fpath):
                    size = os.path.getsize(fpath) / 1e6
                    print(f"      {f} ({size:.1f} MB)")

    # Check for checkpoints
    checkpoints = [d for d in os.listdir(phase_0_5_output) if d.startswith("checkpoint-")]
    if checkpoints:
        print(f"\n   Checkpoints: {len(checkpoints)}")
        for cp in sorted(checkpoints, key=lambda x: int(x.split('-')[1]) if x.split('-')[1].isdigit() else 0):
            print(f"      - {cp}")

    # Load results
    results_path = os.path.join(phase_0_5_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 0.5 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
            elif isinstance(v, dict):
                print(f"      {k}: {v}")
    else:
        # Try trainer_state.json
        trainer_state_path = os.path.join(phase_0_5_output, "best", "trainer_state.json")
        if os.path.exists(trainer_state_path):
            with open(trainer_state_path) as f:
                state = json.load(f)
            print("\nPhase 0.5 Training State (from best model):")
            print(f"      Global step: {state.get('global_step', 'N/A')}")
            print(f"      Best metric: {state.get('best_metric', 'N/A')}")
else:
    print(f"Phase 0.5 output not found at {phase_0_5_output}")

### [SKIP FOR NOW] Benchmark Healed Model on FamilyOS Tasks

**Note:** Phase 0.5 trains on HuggingFace healing data (SST-2, MNLI, CoNLL, STS-B), NOT FamilyOS data.
The v3 model architecture and heads are different from v2, so the v2 benchmark script won't work directly.

**Run FamilyOS benchmark AFTER Phase 1** which trains on actual FamilyOS data.

For now, use the **Healing Validation** cell below to verify Phase 0.5 worked correctly.

---

**Reference - v2 Baseline (checkpoint-18000):** 87.25% weighted average
- Safety: 97.2%, Intent: 93.4%, NER: 87.8%, Ingress: 87.6%

In [ ]:
# =============================================================================
# SKIP FOR PHASE 0.5 - Run after Phase 1
# =============================================================================
# Phase 0.5 uses HuggingFace healing data, not FamilyOS data.
# The v3 model has different architecture than v2, so benchmark needs updating.
#
# For Phase 0.5 validation, use the "Healing Validation" cell below.
# Run FamilyOS benchmark AFTER Phase 1 completes.

print("="*60)
print("FAMILYOS BENCHMARK - SKIPPED FOR PHASE 0.5")
print("="*60)
print()
print("Phase 0.5 trains on HuggingFace healing data:")
print("   - SST-2 (sentiment)")
print("   - MNLI (NLI)")
print("   - CoNLL-2003 (NER)")
print("   - STS-B (similarity)")
print()
print("The v3 model architecture differs from v2:")
print("   - v3 has 28 layers (v2 has 22)")
print("   - v3 uses hub tokens [EMO], [MEM], [REL], [TASK]")
print("   - v3 task heads are saved separately in task_heads.pt")
print()
print("To benchmark FamilyOS performance:")
print("   1. Complete Phase 1 (trains on FamilyOS unified data)")
print("   2. Run FamilyOS benchmark after Phase 1")
print()
print("For now, run the 'Healing Validation' cell below to verify Phase 0.5.")
print("="*60)

### Phase 0.5 Healing Validation

**Critical checks to verify healing succeeded:**
1. **Layer Activation Flow**: L22 -> L23 interface should show smooth activation transfer
2. **Hub Token Embeddings**: [EMO], [MEM], [REL], [TASK] should have learned distinct representations
3. **Base Knowledge Preservation**: Model should still perform well on SST-2, MNLI
4. **Family Layer Coherence**: L23-28 (cloned from L15-20) should produce coherent outputs

In [ ]:
%%time
# =============================================================================
# PHASE 0.5 HEALING VALIDATION - COMPREHENSIVE EVALUATION
# =============================================================================

import os
import sys
import torch
import numpy as np
from pathlib import Path

# Get healed model path - Phase 0.5 saves to 'best/', not 'best_model/'
healed_model_path = None
for model_dir in ["best", "best_model", "final_model"]:
    candidate = f"{OUTPUT_DIR}/phase_0.5/{model_dir}"
    if os.path.exists(candidate):
        healed_model_path = candidate
        break

if not healed_model_path:
    print("ERROR: No healed model found! Run Phase 0.5 first.")
else:
    print("="*70)
    print("PHASE 0.5 HEALING VALIDATION")
    print("="*70)
    print(f"Model: {healed_model_path}")

    # Load model and tokenizer
    sys.path.insert(0, "src")
    from modeling_studio.models.modernbert_v3 import ModernBERTv3Ultra
    from modeling_studio.models.hub_tokens import HUB_TOKEN_IDS
    from transformers import AutoTokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Load healed model
    print("\nLoading healed model...")
    model = ModernBERTv3Ultra.from_pretrained(healed_model_path, device=str(device))
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    # =========================================================================
    # TEST 1: Hub Token Embedding Analysis
    # =========================================================================
    print("\n" + "-"*70)
    print("TEST 1: Hub Token Embedding Analysis")
    print("-"*70)

    # Hub tokens use hardcoded IDs (not in tokenizer vocab)
    hub_tokens = ["[CLS]", "[EMO]", "[MEM]", "[REL]", "[TASK]"]
    hub_ids = [
        tokenizer.cls_token_id,  # [CLS] from tokenizer
        HUB_TOKEN_IDS["[EMO]"],  # 50368
        HUB_TOKEN_IDS["[MEM]"],  # 50369
        HUB_TOKEN_IDS["[REL]"],  # 50370
        HUB_TOKEN_IDS["[TASK]"], # 50371
    ]

    # Get embeddings
    with torch.no_grad():
        embeddings = model.embeddings.word_embeddings.weight[hub_ids].cpu().numpy()

    # Compute pairwise cosine similarities
    from numpy.linalg import norm
    def cosine_sim(a, b):
        return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

    print("\nHub Token Cosine Similarities:")
    print(f"{'':12}", end="")
    for t in hub_tokens:
        print(f"{t:>10}", end="")
    print()

    for i, t1 in enumerate(hub_tokens):
        print(f"{t1:12}", end="")
        for j, t2 in enumerate(hub_tokens):
            sim = cosine_sim(embeddings[i], embeddings[j])
            print(f"{sim:10.3f}", end="")
        print()

    # Check if hub tokens are distinct (not too similar)
    off_diag_sims = []
    for i in range(len(hub_tokens)):
        for j in range(i+1, len(hub_tokens)):
            off_diag_sims.append(cosine_sim(embeddings[i], embeddings[j]))

    avg_sim = np.mean(off_diag_sims)
    max_sim = np.max(off_diag_sims)

    hub_distinct = max_sim < 0.95  # Hub tokens should not be too similar
    print(f"\nAvg off-diagonal similarity: {avg_sim:.3f}")
    print(f"Max off-diagonal similarity: {max_sim:.3f}")
    print(f"Hub tokens distinct: {'PASS' if hub_distinct else 'FAIL'} (max < 0.95)")

    # =========================================================================
    # TEST 2: Layer Activation Flow (L22 -> L23 Interface)
    # =========================================================================
    print("\n" + "-"*70)
    print("TEST 2: Layer Activation Flow (L22 -> L23 Interface)")
    print("-"*70)

    test_texts = [
        "I love this product, it's amazing!",
        "The weather today is quite pleasant.",
        "She went to the store to buy groceries.",
    ]

    # Hook to capture layer outputs
    layer_outputs = {}
    def make_hook(name):
        def hook(module, input, output):
            if isinstance(output, tuple):
                layer_outputs[name] = output[0].detach()
            else:
                layer_outputs[name] = output.detach()
        return hook

    # Register hooks on layers around interface
    hooks = []
    encoder = model.encoder if hasattr(model, 'encoder') else model
    layers = encoder.layers if hasattr(encoder, 'layers') else []

    for idx in [20, 21, 22, 23, 24, 25]:  # Around interface
        if idx < len(layers):
            h = layers[idx].register_forward_hook(make_hook(f"layer_{idx}"))
            hooks.append(h)

    # Run forward pass
    inputs = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        _ = model(**inputs)

    # Remove hooks
    for h in hooks:
        h.remove()

    # Analyze activation flow
    print("\nLayer Activation Statistics (mean, std, max):")
    interface_ratio = None
    for name in sorted(layer_outputs.keys()):
        out = layer_outputs[name]
        mean_val = out.mean().item()
        std_val = out.std().item()
        max_val = out.abs().max().item()
        print(f"  {name}: mean={mean_val:8.4f}, std={std_val:8.4f}, max={max_val:8.4f}")

        # Compute interface ratio (L23/L22)
        if name == "layer_22" and "layer_23" in layer_outputs:
            l22_norm = layer_outputs["layer_22"].norm().item()
            l23_norm = layer_outputs["layer_23"].norm().item()
            interface_ratio = l23_norm / l22_norm if l22_norm > 0 else 0

    interface_smooth = True  # Default
    if interface_ratio:
        interface_smooth = 0.5 < interface_ratio < 2.0
        print(f"\nL23/L22 activation ratio: {interface_ratio:.3f}")
        print(f"Interface smooth: {'PASS' if interface_smooth else 'FAIL'} (0.5 < ratio < 2.0)")

    # =========================================================================
    # TEST 3: Quick SST-2 Sentiment Evaluation
    # =========================================================================
    print("\n" + "-"*70)
    print("TEST 3: SST-2 Sentiment Evaluation (100 samples)")
    print("-"*70)

    from datasets import load_dataset

    # Load SST-2 validation set
    sst2 = load_dataset("glue", "sst2", split="validation[:100]")

    # Check if sentiment head exists
    head_path = os.path.join(healed_model_path, "task_heads.pt")
    sst2_pass = None

    if os.path.exists(head_path):
        heads = torch.load(head_path, map_location=device, weights_only=True)
        if "sentiment_head" in heads:
            sentiment_head = torch.nn.Linear(model.config.hidden_size, 2).to(device)
            sentiment_head.load_state_dict(heads["sentiment_head"])
            sentiment_head.eval()

            correct = 0
            for sample in sst2:
                text = sample["sentence"]
                label = sample["label"]

                inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs)
                    # Use position 1 ([EMO] position) for sentiment
                    emo_hidden = outputs.last_hidden_state[:, 1, :]
                    logits = sentiment_head(emo_hidden)
                    pred = logits.argmax(dim=-1).item()

                if pred == label:
                    correct += 1

            accuracy = correct / len(sst2)
            sst2_pass = accuracy >= 0.70  # 70% threshold for healing
            print(f"SST-2 Accuracy: {accuracy:.1%} ({correct}/{len(sst2)})")
            print(f"SST-2 Threshold: {'PASS' if sst2_pass else 'FAIL'} (>= 70%)")
        else:
            print("Sentiment head not found in checkpoint")
    else:
        print("Task heads not found - using embedding similarity heuristic")

        # Use embedding-based heuristic
        pos_words = ["good", "great", "excellent", "amazing", "love", "wonderful"]
        neg_words = ["bad", "terrible", "awful", "hate", "horrible", "poor"]

        pos_ids = tokenizer.convert_tokens_to_ids(pos_words)
        neg_ids = tokenizer.convert_tokens_to_ids(neg_words)

        with torch.no_grad():
            pos_emb = model.embeddings.word_embeddings.weight[pos_ids].mean(dim=0)
            neg_emb = model.embeddings.word_embeddings.weight[neg_ids].mean(dim=0)

        correct = 0
        for sample in sst2:
            text = sample["sentence"]
            label = sample["label"]

            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                cls_hidden = outputs.last_hidden_state[:, 0, :].squeeze()

                pos_sim = torch.cosine_similarity(cls_hidden, pos_emb, dim=0)
                neg_sim = torch.cosine_similarity(cls_hidden, neg_emb, dim=0)
                pred = 1 if pos_sim > neg_sim else 0

            if pred == label:
                correct += 1

        accuracy = correct / len(sst2)
        sst2_pass = accuracy >= 0.55  # Lower threshold for heuristic
        print(f"SST-2 Accuracy (heuristic): {accuracy:.1%} ({correct}/{len(sst2)})")
        print(f"SST-2 Threshold: {'PASS' if sst2_pass else 'FAIL'} (>= 55% for heuristic)")

    # =========================================================================
    # TEST 4: Family Layer Coherence (L23-28)
    # =========================================================================
    print("\n" + "-"*70)
    print("TEST 4: Family Layer Coherence (L23-28)")
    print("-"*70)

    # Check gradient norms during a forward-backward pass
    model.train()

    test_input = tokenizer("This is a test sentence for checking gradients.",
                           return_tensors="pt", truncation=True)
    test_input = {k: v.to(device) for k, v in test_input.items()}

    outputs = model(**test_input)
    loss = outputs.last_hidden_state.mean()  # Dummy loss
    loss.backward()

    # Check gradient norms for family layers
    print("\nGradient Norms by Layer Band:")
    grad_norms = {"foundation": [], "core": [], "semantic": [], "family": []}

    for name, param in model.named_parameters():
        if param.grad is not None and "layers" in name:
            # Extract layer number
            try:
                layer_num = int(name.split("layers.")[1].split(".")[0])
                grad_norm = param.grad.norm().item()

                if layer_num <= 5:
                    grad_norms["foundation"].append(grad_norm)
                elif layer_num <= 17:
                    grad_norms["core"].append(grad_norm)
                elif layer_num <= 21:
                    grad_norms["semantic"].append(grad_norm)
                else:
                    grad_norms["family"].append(grad_norm)
            except:
                pass

    for band, norms in grad_norms.items():
        if norms:
            avg = np.mean(norms)
            print(f"  {band:12}: avg_grad_norm = {avg:.6f} ({len(norms)} params)")

    # Family layers should have non-zero gradients (they're trainable)
    family_has_grads = len(grad_norms["family"]) > 0 and np.mean(grad_norms["family"]) > 1e-10
    print(f"\nFamily layers have gradients: {'PASS' if family_has_grads else 'FAIL'}")

    model.zero_grad()
    model.eval()

    # =========================================================================
    # SUMMARY
    # =========================================================================
    print("\n" + "="*70)
    print("HEALING VALIDATION SUMMARY")
    print("="*70)

    results = {
        "Hub Tokens Distinct": hub_distinct,
        "Interface Smooth (L22->L23)": interface_smooth if interface_ratio else "N/A",
        "SST-2 Performance": sst2_pass if sst2_pass is not None else "N/A",
        "Family Layers Active": family_has_grads,
    }

    all_pass = all(v == True for v in results.values() if v != "N/A")

    for test, result in results.items():
        status = "PASS" if result == True else ("FAIL" if result == False else result)
        print(f"  {test}: {status}")

    print("\n" + "-"*70)
    if all_pass:
        print("HEALING VALIDATION: PASSED")
        print("The model is ready for Phase 1 training!")
    else:
        print("HEALING VALIDATION: SOME TESTS FAILED")
        print("Review the failed tests before proceeding to Phase 1.")
    print("-"*70)

---
## 6. Phase 1: Multi-Task FamilyOS Training

**Purpose:** Main training on FamilyOS multi-task data.

**Data:** Local FamilyOS datasets:
- Emotions: `data/familyos/emotions/silver`
- Temporal: `data/familyos/temporal/silver`
- Unified: `data/familyos/unified/output`

**Training:** 10,000 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_1:
    print("="*60)
    print("PHASE 1: Multi-Task FamilyOS")
    print("="*60)

    # Get model from Phase 0.5 - Phase 0.5 saves to 'best/', not 'best_model/'
    model_path = None
    for model_dir in ["best", "best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_0.5/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 0.5 found!")
        print("Expected one of:")
        print(f"   - {OUTPUT_DIR}/phase_0.5/best/")
        print(f"   - {OUTPUT_DIR}/phase_0.5/best_model/")
        print(f"   - {OUTPUT_DIR}/phase_0.5/final_model/")
        phase_1_ok = False
    else:
        print(f"Using model: {model_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/train_v3_phase1.py "
        cmd += f"--config configs/training/multitask/stage_v3_phase1.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_1 "
        cmd += f"--model-path {model_path} "

        if DEBUG_RUN:
            cmd += "--max-steps 5 --debug "

        if not USE_WANDB:
            cmd += "--no-wandb "
        else:
            cmd += "--wandb-run-name v3_phase_1 "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check success
        phase_1_ok = os.path.exists(f"{OUTPUT_DIR}/phase_1/final_model/pytorch_model.bin") or \
                     os.path.exists(f"{OUTPUT_DIR}/phase_1/best_model/pytorch_model.bin")

        if phase_1_ok:
            print("\n" + "="*60)
            print("PHASE 1 COMPLETED SUCCESSFULLY!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 1 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 1 skipped (RUN_PHASE_1 = False)")
    phase_1_ok = os.path.exists(f"{OUTPUT_DIR}/phase_1/best_model")

In [ ]:
# Verify Phase 1 output
import os
import json

phase_1_output = f"{OUTPUT_DIR}/phase_1"

if os.path.exists(phase_1_output):
    print(f"Phase 1 output: {phase_1_output}")

    # Check for model
    for model_dir in ["best_model", "final_model"]:
        model_path = os.path.join(phase_1_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model: {model_dir}/")
            for f in os.listdir(model_path):
                size = os.path.getsize(os.path.join(model_path, f)) / 1e6
                print(f"      {f} ({size:.1f} MB)")

    # Load results
    results_path = os.path.join(phase_1_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 1 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"Phase 1 output not found at {phase_1_output}")

---
## 7. Phase 1.5: Forgetting Gate Evaluation

**Purpose:** Verify the model hasn't catastrophically forgotten base knowledge.

**Evaluation:**
- Compares Phase 1 model to Phase 0.5 baseline
- Tests on SST-2, Civil Comments, etc.
- Must pass <2% accuracy drop threshold

In [ ]:
%%time

import os

if RUN_PHASE_1_5:
    print("="*60)
    print("PHASE 1.5: Forgetting Gate Evaluation")
    print("="*60)

    # Get model from Phase 1
    model_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_1/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    # Get baseline from Phase 0.5
    baseline_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_0.5/{model_dir}"
        if os.path.exists(candidate):
            baseline_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 1 found!")
        phase_1_5_ok = False
    else:
        print(f"Evaluating model: {model_path}")
        if baseline_path:
            print(f"Baseline model: {baseline_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/evaluate_forgetting.py "
        cmd += f"--config configs/evaluation/forgetting_gate.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_1.5 "
        cmd += f"--model-path {model_path} "

        if baseline_path:
            cmd += f"--baseline {baseline_path} "

        if not USE_WANDB:
            cmd += "--no-wandb "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check results
        results_path = f"{OUTPUT_DIR}/phase_1.5/results.json"
        phase_1_5_ok = os.path.exists(results_path)

        if phase_1_5_ok:
            print("\n" + "="*60)
            print("PHASE 1.5 COMPLETED!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 1.5 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 1.5 skipped (RUN_PHASE_1_5 = False)")
    phase_1_5_ok = True

In [ ]:
# Check Forgetting Gate Results
import os
import json

phase_1_5_output = f"{OUTPUT_DIR}/phase_1.5"
results_path = os.path.join(phase_1_5_output, "results.json")

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

    print("="*60)
    print("FORGETTING GATE RESULTS")
    print("="*60)

    # Check if gate passed
    gate_passed = results.get("gate_passed", True)

    if "forgetting_metrics" in results:
        print("\nForgetting by Task (max allowed: 2%):")
        for task, drop in results["forgetting_metrics"].items():
            status = "PASS" if drop <= 0.02 else "FAIL"
            print(f"   [{status}] {task}: {drop:.2%}")

    if gate_passed:
        print("\n" + "="*60)
        print("FORGETTING GATE PASSED! Proceeding to Phase 2.")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("FORGETTING GATE FAILED!")
        print("Consider: Increase replay ratio and re-run Phase 1")
        print("="*60)
else:
    print(f"No forgetting results found at {results_path}")
    print("Assuming gate passed (no evaluation run).")

---
## 8. Phase 2: Fine-Tuning

**Purpose:** Final refinement and head optimization.

**Training:**
- Lower learning rate
- Focus on task-specific heads
- 5,000 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_2:
    print("="*60)
    print("PHASE 2: Fine-Tuning")
    print("="*60)

    # Get model from Phase 1 (not 1.5, which is evaluation only)
    model_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_1/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 1 found!")
        phase_2_ok = False
    else:
        print(f"Using model: {model_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/train_v3_phase2.py "
        cmd += f"--config configs/training/multitask/stage_v3_phase2.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_2 "
        cmd += f"--model-path {model_path} "

        if DEBUG_RUN:
            cmd += "--max-steps 5 --debug "

        if not USE_WANDB:
            cmd += "--no-wandb "
        else:
            cmd += "--wandb-run-name v3_phase_2 "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check success
        phase_2_ok = os.path.exists(f"{OUTPUT_DIR}/phase_2/final_model/pytorch_model.bin") or \
                     os.path.exists(f"{OUTPUT_DIR}/phase_2/best_model/pytorch_model.bin")

        if phase_2_ok:
            print("\n" + "="*60)
            print("PHASE 2 COMPLETED SUCCESSFULLY!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 2 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 2 skipped (RUN_PHASE_2 = False)")
    phase_2_ok = os.path.exists(f"{OUTPUT_DIR}/phase_2/best_model")

In [ ]:
# Verify Phase 2 output
import os
import json

phase_2_output = f"{OUTPUT_DIR}/phase_2"

if os.path.exists(phase_2_output):
    print(f"Phase 2 output: {phase_2_output}")

    # Check for model
    for model_dir in ["best_model", "final_model"]:
        model_path = os.path.join(phase_2_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model: {model_dir}/")
            for f in os.listdir(model_path):
                size = os.path.getsize(os.path.join(model_path, f)) / 1e6
                print(f"      {f} ({size:.1f} MB)")

    # Load results
    results_path = os.path.join(phase_2_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 2 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"Phase 2 output not found at {phase_2_output}")

---
## 9. Training Summary

In [ ]:
import os
import json
from datetime import datetime

print("="*60)
print("MODERNBERT V3 TRAINING SUMMARY")
print("="*60)
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check all phases
phases = {
    "Phase 0.5 (Healing)": f"{OUTPUT_DIR}/phase_0.5",
    "Phase 1 (Multi-Task)": f"{OUTPUT_DIR}/phase_1",
    "Phase 1.5 (Forgetting)": f"{OUTPUT_DIR}/phase_1.5",
    "Phase 2 (Fine-Tuning)": f"{OUTPUT_DIR}/phase_2",
}

print("\nPhase Status:")
for name, path in phases.items():
    # Check for model or results
    has_model = os.path.exists(f"{path}/best_model") or os.path.exists(f"{path}/final_model")
    has_results = os.path.exists(f"{path}/results.json")

    if has_model or has_results:
        status = "Complete"
    else:
        status = "Not Run"

    print(f"   {name}: {status}")

# Find final model
final_model = None
for phase in ["phase_2", "phase_1", "phase_0.5"]:
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/{phase}/{model_dir}"
        if os.path.exists(candidate):
            final_model = candidate
            break
    if final_model:
        break

print("\n" + "="*60)
print("FINAL MODEL")
print("="*60)
if final_model:
    print(f"   {final_model}")

    # Count parameters
    model_file = os.path.join(final_model, "pytorch_model.bin")
    if os.path.exists(model_file):
        size_mb = os.path.getsize(model_file) / 1e6
        print(f"   Size: {size_mb:.1f} MB")
else:
    print("   No model found!")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("   1. Backup model to Google Drive")
print("   2. Run final evaluation on held-out test set")
print("   3. Export to ONNX for inference")
print("   4. Deploy to production")
print("="*60)

---
## 10. Backup to Google Drive

In [ ]:
# Backup outputs to Google Drive
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT_v3/runs/{timestamp}"

    print(f"Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy each phase
    for phase in ["phase_0.5", "phase_1", "phase_1.5", "phase_2"]:
        phase_path = f"{OUTPUT_DIR}/{phase}"
        if os.path.exists(phase_path):
            shutil.copytree(
                phase_path,
                f"{backup_dir}/{phase}",
                dirs_exist_ok=True
            )
            print(f"   Backed up: {phase}")

    print(f"\nBackup complete: {backup_dir}")
else:
    print("Not on Colab - skipping Drive backup.")
    print(f"Outputs are in: {OUTPUT_DIR}")

In [ ]:
# Empty cell for notes